# 03 — Model comparison, threshold optimization & SHAP
### CREDIT CARD FRAUD DETECTION SYSTEM

This notebook digs into the trained model: threshold selection strategies, their consequences on the test set, and the SHAP global explanation produced by `TreeExplainer`.

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path('backend').resolve()))

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

from app.fraud_detector.config import DEFAULT_MODEL_CONFIG, TARGET
from app.fraud_detector.data.loading import load_dataset
from app.fraud_detector.features.build import FeaturePreprocessor, engineer_features
from app.fraud_detector.models.xgboost_model import XGBFraudModel
from app.fraud_detector.evaluation.metrics import compute_metrics
from app.fraud_detector.evaluation.threshold import threshold_sweep
from app.fraud_detector.utils.artifacts import active_version, read_json, version_dir

# Load the ACTIVE trained model + preprocessor from artifacts
version = active_version()
assert version, "Train first: python train_model.py"
model = XGBFraudModel.load(version_dir(version) / 'model.joblib')
pp = FeaturePreprocessor.load(Path('../artifacts/preprocessing/preprocessor.joblib'))
thresholds = read_json(version_dir(version) / 'thresholds.json')
print('Active model:', version)
print('Thresholds:', {k: round(v, 4) if isinstance(v, float) else v for k, v in thresholds.items()})

In [ ]:
df, _ = load_dataset()
X_raw, y = df.drop(columns=[TARGET]), df[TARGET].to_numpy()
idx_train, idx_test = next(StratifiedShuffleSplit(n_splits=1, test_size=DEFAULT_MODEL_CONFIG.test_size, random_state=42).split(X_raw, y))
X_test = pp.transform(engineer_features(df.iloc[idx_test]))
y_test = y[idx_test]
proba = model.predict_proba(X_test)

print('Test set size:', len(y_test), '| frauds:', int(y_test.sum()))
sweep = threshold_sweep(y_test, proba, n_points=60)
best = sweep.loc[sweep['f1'].idxmax()]
print('Best F1 point on TEST (for reference): threshold %.3f -> precision %.3f recall %.3f f1 %.3f'
      % (best['threshold'], best['precision'], best['recall'], best['f1']))
t = thresholds['best_threshold']
m = compute_metrics(y_test, proba, t)
print('ACTIVE threshold %.3f -> precision %.3f recall %.3f f1 %.3f PR-AUC %.3f'
      % (t, m['precision'], m['recall'], m['f1'], m['pr_auc']))
print('Confusion matrix:', m['confusion_matrix'])

In [ ]:
import shap
import numpy as np

background = np.load(Path('../artifacts/preprocessing/shap_background.npy'))
explainer = shap.TreeExplainer(model.estimator, data=background, model_output='probability')
sample = X_test[:500]
values = explainer.shap_values(sample)

from app.fraud_detector.features.build import FEATURE_COLUMNS
mean_abs = np.abs(values).mean(axis=0)
order = np.argsort(mean_abs)[::-1]
print('Top-10 mean |SHAP| contributions (global importance):')
for i in order[:10]:
    print(f'  {FEATURE_COLUMNS[i]:<14} {mean_abs[i]:.4f}')

In [ ]:
# Local explanation for the single highest-scoring test row
row_idx = int(np.argmax(proba))
row_values = explainer.shap_values(X_test[row_idx:row_idx + 1])
contrib = sorted(zip(FEATURE_COLUMNS, row_values[0]), key=lambda t: abs(t[1]), reverse=True)
print(f'Row {row_idx}: model probability = {proba[row_idx]:.4f}')
print('Top contributions:')
for name, v in contrib[:8]:
    print(f'  {name:<14} {v:+.4f}  {"pushes UP (fraud)" if v > 0 else "pushes DOWN (legit)"}')

## What this comparison shows
- Threshold selection is a **business decision**, not a mathematical constant: the same model produces very different precision/recall pairs at different cut points.
- SHAP contributions decompose the model score for each row; their sum plus the base value reconstructs the probability.
- Because features are PCA-transformed, SHAP tells you *which inputs the model relied on* — it cannot name the real-world behaviour behind a signal, and it is not evidence of causality.